# Silver Layer — Metadata Transformation
**Layer:** Silver (cleaned, standardized, validated)
**Source:** bronze.bronze_metadata
**Purpose:** Clean and validate Bronze data. Valid rows → silver_metadata. Invalid rows → bronze_metadata_quarantine.

In [ ]:
# Catalog widget — injected by Job base_parameters or set manually
dbutils.widgets.text("catalog", "metadata_governance", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

## Step 1 — Read Metadata from Bronze Layer

In [ ]:
bronze_df = spark.table(f"{catalog}.bronze.bronze_metadata")
bronze_count = bronze_df.count()
print(f"Bronze rows loaded: {bronze_count}")
assert bronze_count > 0, "Bronze table is empty — cannot proceed with Silver transformation"

## Step 2 — Clean and Standardize Metadata

In [ ]:
from pyspark.sql.functions import col, trim, lower

text_cols = [
    "column_name", "column_desc", "term_name", "term_description",
    "security_classification", "term_subdomain", "data_steward",
    "table_name", "table_desc", "table_owner_in_source", "schema_name",
    "database_name", "system_name", "location", "tag_name",
    "tag_value", "certification_level"
]

df_cleaned = bronze_df
for c in text_cols:
    if c in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(c, lower(trim(col(c))))

print(f"Cleaned rows: {df_cleaned.count()}")

## Step 3 — Apply Data Quality Validation Rules

In [ ]:
from pyspark.sql.functions import isnull, when, concat_ws, lit

df_invalid = df_cleaned.filter(
    isnull(col("table_name")) | (col("table_name") == "") |
    isnull(col("schema_name")) | (col("schema_name") == "") |
    isnull(col("database_name")) | (col("database_name") == "")
).withColumn(
    "failure_reason",
    concat_ws(", ",
        when(isnull(col("table_name")) | (col("table_name") == ""), lit("table_name is null")),
        when(isnull(col("schema_name")) | (col("schema_name") == ""), lit("schema_name is null")),
        when(isnull(col("database_name")) | (col("database_name") == ""), lit("database_name is null"))
    )
)

df_valid = df_cleaned.filter(
    col("table_name").isNotNull() & (col("table_name") != "") &
    col("schema_name").isNotNull() & (col("schema_name") != "") &
    col("database_name").isNotNull() & (col("database_name") != "")
)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
print(f"Valid rows:   {valid_count}")
print(f"Invalid rows: {invalid_count}")
assert valid_count + invalid_count == bronze_count, "Row count mismatch"

## Step 4 — Save Validated Metadata to Silver Layer

In [ ]:
df_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.silver_metadata")

print(f"Silver table written: {valid_count} rows")

## Step 5 — Save Quarantine Records and Generate Compliance Alert

In [ ]:
if invalid_count > 0:
    df_invalid.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog}.silver.bronze_metadata_quarantine")

    print(f"Alert: {invalid_count} rows moved to quarantine!")
    display(
        df_invalid.groupBy("failure_reason")
        .count()
        .orderBy("count", ascending=False)
    )
    today = spark.sql("SELECT current_date()").collect()[0][0]
    print(f"[COMPLIANCE ALERT] {invalid_count} rows failed validation. Table: {catalog}.silver.bronze_metadata_quarantine. Date: {today}")
else:
    print("All rows passed validation")

print("Silver notebook execution completed successfully.")

## Step 6 — Validate Silver Output

In [ ]:
silver_df = spark.table(f"{catalog}.silver.silver_metadata")
print(f"Silver final row count: {silver_df.count()}")
silver_df.printSchema()
display(silver_df.limit(5))